# 第一次用 PyTorch：把同一个网络对照、训练、保存下来

[本周导航](README.md) · [前一本：NumPy 网络](01_numpy_network_walkthrough.ipynb) · [环境与内核](../../../docs/setup/DEEP_LEARNING.md)

对应 **W3-08～W3-10，建议分 3 次完成，每次 90～120 分钟**。第一次跟着本书跑通演示，第二次遮住代码做独立练习，最后在 `work/intro-network/network.ipynb` 写自己的解释。这里只需要 CPU，digits 数据随 scikit-learn 提供，无需下载。

**前置条件：** 已理解 `X @ W + b → ReLU → logits → 交叉熵`，且前一本的 NumPy 梯度检查通过。程序上本书可以单独从新内核运行：会导入同目录的 `numpy_mlp.py`，重新建立固定的数据划分，不依赖前一本内存里的变量。

| 本次停靠点 | 先读这里的必要概念 | 跟做后留下的证据 | 需要时再查材料 |
|---|---|---|---|
| [W3-08](#w3-08) | Tensor、计算图、权重转置、logits | 前向／梯度／一步更新对照表 | [神经网络 §1～3](../../materials/10_neural_networks_and_backprop.md) |
| [W3-09](#w3-09) | batch、epoch、梯度清零、验证集选模型 | 两条训练曲线、checkpoint、冻结配置 | [训练模式 §1～2](../../materials/11_deep_learning_patterns.md) |
| [W3-10](#w3-10) | state_dict、预处理复用、最终测试 | 重载一致性、逐类指标、错例 | 本书最后的复盘模板 |

**怎样使用代码：** 标为“演示”的单元可直接运行，注释解释每一步的目的；“独立练习”不参与演示的变量流，尚未填写也能 Run All。完成练习以你自己的推理与代码为准，不能只凭演示运行成功勾选过关。

输出统一写入 `week3/outputs/pytorch_tutorial/`。重跑会覆盖这个演示目录中的同名结果；个人试验请另存到 `work/intro-network/`。


## 0｜先确认根目录与环境

从根目录、本周目录或 `work/intro-network/` 启动都能自动找到项目。环境问题先按[深度学习环境指南](../../../docs/setup/DEEP_LEARNING.md)设置并选择 **ML · PyTorch (.venv-dl)** 内核，再“重启内核并运行全部”。这里故意不自动安装、不跳过 PyTorch；缺依赖必须先解决。


In [ ]:
from pathlib import Path
import sys
import json
import platform

# 从当前工作目录向上找项目，避免把自己的电脑绝对路径写死。
ROOT = next((p for p in [Path.cwd(), *Path.cwd().parents]
             if (p / "course/bootcamp").is_dir()
             and (p / "requirements.txt").is_file()), None)
if ROOT is None:
    raise RuntimeError("请从项目根目录或其子目录启动 Jupyter；参考 docs/setup/DEEP_LEARNING.md。")
WEEK3 = ROOT / "course/bootcamp/week3"
OUT = WEEK3 / "outputs/pytorch_tutorial"
OUT.mkdir(parents=True, exist_ok=True)
print("项目：", ROOT)
print("Python：", sys.executable)


In [ ]:
try:
    import numpy as np
    import pandas as pd
    import matplotlib.pyplot as plt
    import sklearn
    import torch
    from torch import nn
    from sklearn.datasets import load_digits
    from sklearn.model_selection import train_test_split
    from sklearn.preprocessing import StandardScaler
    from sklearn.linear_model import LogisticRegression
    from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
    from sklearn.metrics import ConfusionMatrixDisplay
    from IPython.display import display
except ImportError as exc:
    raise ImportError(
        "缺少本课依赖。按 docs/setup/DEEP_LEARNING.md 创建 .venv-dl，"
        "选 ML · PyTorch (.venv-dl) 内核后重启并 Run All。"
    ) from exc

sys.path.insert(0, str(WEEK3))
from numpy_mlp import init_params, forward, softmax_cross_entropy
from numpy_mlp import loss_and_gradients, sgd_step

SEED = 42
DTYPE = torch.float64  # 对照时保留更多有效数字；输入和模型必须同 dtype。
DEVICE = torch.device("cpu")
torch.manual_seed(SEED)
torch.set_num_threads(1)  # 这个小模型单线程足够；也减少运行时间波动。
torch.use_deterministic_algorithms(True)
print({"numpy": np.__version__, "sklearn": sklearn.__version__, "torch": torch.__version__})


<a id="w3-08"></a>
## W3-08｜用 PyTorch 重新算一遍已经理解的网络

### 1｜Tensor 和自动求导没有改变数学

Tensor 可先理解成带着 dtype、device 等信息的多维数组。`requires_grad=True` 表示要追踪这个参数参与的运算；`backward()` 沿实际执行的计算图运用链式法则，把梯度累加到参数的 `.grad`。它不自动更新参数。

回忆 `L(w)=(2w-3)²`：`dL/dw=4(2w-3)`。从 `w=0` 开始，loss 为 9，梯度为 -12；学习率 0.1 的一步更新应该得到 `w=1.2`。先预测，再运行。[参考：PyTorch 自动求导](https://docs.pytorch.org/tutorials/beginner/basics/autogradqs_tutorial.html)


In [ ]:
# 演示：标量也是 Tensor，shape 是 ()。
w = torch.tensor(0.0, dtype=DTYPE, requires_grad=True)
loss = (2 * w - 3) ** 2
loss.backward()
print("shape / loss / grad:", w.shape, loss.item(), w.grad.item())
assert loss.item() == 9.0 and w.grad.item() == -12.0

with torch.no_grad():  # 更新本身不应该成为下一次反传要追踪的计算图。
    w -= 0.1 * w.grad
assert np.isclose(w.item(), 1.2)
print("更新后的 w:", w.item())


### 2｜Linear 的权重为什么要转置

我们在 NumPy 写 `X @ W + b`，`W` 的 shape 是 `(输入维数, 输出维数)`。`nn.Linear(in_features, out_features)` 保存的 `weight` 是 `(输出维数, 输入维数)`，内部计算相当于 `X @ weight.T + bias`。同一个网络因此复制的是 `W.T`，不是改变公式。[参考：Linear](https://docs.pytorch.org/docs/stable/generated/torch.nn.Linear.html)

| 量 | 本例 shape | 含义 |
|---|---|---|
| X | `(5, 3)` | 5 个样本，每个 3 维 |
| NumPy W1 / PyTorch 第一层 weight | `(3, 4)` / `(4, 3)` | 输入到 4 个隐藏单元 |
| 隐藏输出 | `(5, 4)` | 每个样本 4 个激活值 |
| NumPy W2 / PyTorch 第二层 weight | `(4, 2)` / `(2, 4)` | 4 个隐藏单元到 2 个类别 |
| logits | `(5, 2)` | 每个类别一个原始分数 |

下面两个短函数只负责“搭结构”和“复制已有参数”。`copy_` 才是复制数值；NumPy 转 Tensor 后也先复制，避免共享底层数组造成隐蔽的相互修改。


In [ ]:
def make_model(input_dim=64, hidden_dim=32, output_dim=10):
    # Sequential 按顺序调用层，索引 0 和 2 是两层 Linear。
    return nn.Sequential(
        nn.Linear(input_dim, hidden_dim),
        nn.ReLU(),
        nn.Linear(hidden_dim, output_dim),
    ).to(device=DEVICE, dtype=DTYPE)


def copy_numpy_params(model, params):
    with torch.no_grad():
        model[0].weight.copy_(torch.tensor(params["W1"].T, dtype=DTYPE))
        model[0].bias.copy_(torch.tensor(params["b1"], dtype=DTYPE))
        model[2].weight.copy_(torch.tensor(params["W2"].T, dtype=DTYPE))
        model[2].bias.copy_(torch.tensor(params["b2"], dtype=DTYPE))


def as_numpy(tensor):
    # detach 不求梯度，cpu 放到 CPU，copy 防止与原 Tensor 共享数据。
    return tensor.detach().cpu().numpy().copy()


In [ ]:
rng_demo = np.random.default_rng(SEED)
X_demo = rng_demo.normal(size=(5, 3)).astype(np.float64)
y_demo = np.array([0, 1, 0, 1, 1], dtype=np.int64)
params_demo = init_params(3, 4, 2, seed=SEED)
model_demo = make_model(3, 4, 2)
copy_numpy_params(model_demo, params_demo)

X_demo_t = torch.tensor(X_demo, dtype=DTYPE)
y_demo_t = torch.tensor(y_demo, dtype=torch.long)  # 类别下标使用整数类型。
logits_np, _ = forward(X_demo, params_demo)
logits_t = model_demo(X_demo_t)
print("NumPy W1:", params_demo["W1"].shape)
print("PyTorch weight:", tuple(model_demo[0].weight.shape))
print("logits:", tuple(logits_t.shape))


### 3｜训练损失用 logits；展示概率时才 softmax

logits 不要求在 0～1 内，也不要求一行和为 1。softmax 才把一行分数转换成各类概率。`CrossEntropyLoss` 内部已经完成稳定的 log-softmax 和负对数似然计算；如果先 softmax 再传进去，会把“概率”再次当“分数”，变成另一个损失。

这里 `reduction="mean"` 对 batch 求平均，与 NumPy 版本一致。`argmax(logits)` 和 `argmax(softmax(logits))` 一样，所以只预测类别时无需 softmax。[参考：CrossEntropyLoss](https://docs.pytorch.org/docs/stable/generated/torch.nn.CrossEntropyLoss.html)


In [ ]:
criterion = nn.CrossEntropyLoss(reduction="mean")
loss_np, probabilities_np = softmax_cross_entropy(logits_np, y_demo)
loss_t = criterion(logits_t, y_demo_t)  # 正确：直接传 logits。
probabilities_t = torch.softmax(logits_t, dim=1)
assert torch.allclose(probabilities_t.sum(dim=1), torch.ones(5, dtype=DTYPE))
assert torch.equal(logits_t.argmax(dim=1), probabilities_t.argmax(dim=1))
print("原始分数:", as_numpy(logits_t)[0])
print("展示用概率:", as_numpy(probabilities_t)[0])
print("平均交叉熵:", loss_t.item())


### 4｜逐项核对：前向 → loss → 反传 → 更新

**先声明标准，再看结果：** 本次 CPU / float64 对照使用 `atol=1e-8, rtol=1e-6`。`allclose` 判断的是每个元素差异是否在 `atol + rtol × |参考值|` 内。表中同时显示最大绝对差；不能只因 loss 接近就认为全部梯度正确。

`zero_grad` 清掉上次累积梯度，`backward` 计算本次梯度，`step` 才使用优化器更新参数。下面先反传并保存对照表，最后才更新两套参数。


In [ ]:
ATOL, RTOL = 1e-8, 1e-6
comparison_rows = []

def compare(name, expected, actual):
    expected, actual = np.asarray(expected), np.asarray(actual)
    assert expected.shape == actual.shape, f"{name}: shape 不一致"
    passed = np.allclose(expected, actual, atol=ATOL, rtol=RTOL)
    comparison_rows.append({"item": name,
                            "max_abs_diff": float(np.max(np.abs(expected - actual))),
                            "passed": bool(passed)})
    np.testing.assert_allclose(actual, expected, atol=ATOL, rtol=RTOL, err_msg=name)

compare("forward_logits", logits_np, as_numpy(logits_t))
compare("probabilities", probabilities_np, as_numpy(probabilities_t))
compare("mean_cross_entropy", loss_np, loss_t.item())

loss_from_backprop, grads_np = loss_and_gradients(X_demo, y_demo, params_demo)
compare("numpy_loss_path", loss_np, loss_from_backprop)
optimizer_demo = torch.optim.SGD(model_demo.parameters(), lr=0.1)
optimizer_demo.zero_grad(set_to_none=True)
loss_t.backward()


In [ ]:
# 参数与梯度都按 NumPy 的排列方式核对；Linear 的 weight.grad 也要转置。
grads_t = {
    "W1": as_numpy(model_demo[0].weight.grad).T,
    "b1": as_numpy(model_demo[0].bias.grad),
    "W2": as_numpy(model_demo[2].weight.grad).T,
    "b2": as_numpy(model_demo[2].bias.grad),
}
for name in params_demo:
    compare(f"gradient_{name}", grads_np[name], grads_t[name])

# SGD 的两边从完全相同的参数出发，各更新一次。
sgd_step(params_demo, grads_np, lr=0.1)
optimizer_demo.step()
updated_t = {
    "W1": as_numpy(model_demo[0].weight).T,
    "b1": as_numpy(model_demo[0].bias),
    "W2": as_numpy(model_demo[2].weight).T,
    "b2": as_numpy(model_demo[2].bias),
}
for name in params_demo:
    compare(f"one_sgd_step_{name}", params_demo[name], updated_t[name])
comparison_table = pd.DataFrame(comparison_rows)
comparison_table.to_csv(OUT / "numpy_pytorch_comparison.csv", index=False)
display(comparison_table)


**预期现象：** 全部 `passed=True`，差异通常接近浮点舍入量级。若失败，先查 dtype、weight 转置、loss 的 mean / sum、是否额外 softmax、是否重复 backward；不要直接把容差调大。

### 停下来独立练习 A（不影响 Run All）

1. 在自己的 Notebook 把小模型改为 `3→5→3`，自行调整标签与参数，对照四类证据。
2. 不看代码，写出 `zero_grad → forward → loss → backward → step` 每一步改变了什么。
3. 新建一个标量 `w`，重新构造相同公式并连续反传两次但不清梯度，预测 `.grad` 会怎样；随后加入清零核对。

<details><summary>提示（先尝试再展开）</summary>

A1：第一层 weight shape 应为 `(5, 3)`，第二层为 `(3, 5)`；标签只能取 0、1、2。A3：每次重新写 loss 以建立新图；梯度会累加。`backward()` 不会修改 w 的数值，SGD 才修改参数。`w.grad=None` 可清除标量的旧梯度。

</details>

**本次完成标准：** 保存对照表，能解释每一次转置和每一个更新步骤。不能解释时先停在这里。


In [ ]:
# 独立练习区：请在自己的副本中完成，不修改上面的演示变量。
# TODO：写自己的 3→5→3 对照与梯度累加例子。


<a id="w3-09"></a>
## W3-09｜在 digits 上只比较两个学习率

### 5｜划分索引，预处理只从训练集学习

每张 digits 图像为 8×8，展开成 64 维向量，标签为 0～9。我们复用 W3-07 的规则：种子 42、分层约 60% / 20% / 20%；由于样本数不是 5 的倍数，数量不会恰好整齐。

先保存**原始样本索引**，便于追溯错例。第一步拿出 40% 临时集，再把临时集对半分为 validation / test。分层仅使用标签保持类比例；测试图像不用于训练、可视化排错或调参。标准化用训练集的每列均值和标准差，validation / test 只套用这些数。常量列的 `scale_` 由 StandardScaler 设为 1，避免除以 0。


In [ ]:
digits = load_digits()
X_raw = digits.data.astype(np.float64)
y = digits.target.astype(np.int64)
all_idx = np.arange(len(y))
train_idx, heldout_idx = train_test_split(
    all_idx, test_size=0.4, random_state=SEED, stratify=y)
val_idx, test_idx = train_test_split(
    heldout_idx, test_size=0.5, random_state=SEED, stratify=y[heldout_idx])

assert len(set(train_idx) & set(val_idx)) == 0
assert len(set(train_idx) & set(test_idx)) == 0
assert len(set(val_idx) & set(test_idx)) == 0
assert len(np.unique(np.concatenate([train_idx, val_idx, test_idx]))) == len(y)
scaler = StandardScaler().fit(X_raw[train_idx])  # 唯一的 fit 发生在训练集。
X_train = scaler.transform(X_raw[train_idx])
X_val = scaler.transform(X_raw[val_idx])
y_train, y_val = y[train_idx], y[val_idx]
print("train / validation / test 数量:", len(train_idx), len(val_idx), len(test_idx))


In [ ]:
# 本书不要求先运行 NumPy Notebook；若已有它的结果，额外核对划分确实一致。
previous_split = WEEK3 / "outputs/numpy_tutorial/digits_split_and_scaler.npz"
if previous_split.exists():
    with np.load(previous_split, allow_pickle=False) as saved:
        for name, current in [("train_idx", train_idx), ("val_idx", val_idx), ("test_idx", test_idx)]:
            np.testing.assert_array_equal(saved[name], current)
        np.testing.assert_allclose(saved["mean"], scaler.mean_)
        np.testing.assert_allclose(saved["scale"], scaler.scale_)
    print("已与 W3-07 保存的划分及预处理核对一致。")

np.savez(OUT / "digits_split_and_scaler.npz",
         train_idx=train_idx, val_idx=val_idx, test_idx=test_idx,
         mean=scaler.mean_, scale=scaler.scale_)
X_train_t = torch.tensor(X_train, dtype=DTYPE, device=DEVICE)
X_val_t = torch.tensor(X_val, dtype=DTYPE, device=DEVICE)
y_train_t = torch.tensor(y_train, dtype=torch.long, device=DEVICE)
y_val_t = torch.tensor(y_val, dtype=torch.long, device=DEVICE)


### 6｜先建立一个有意义的 baseline

逻辑回归也输出分类分数，但没有隐藏层。本例固定默认正则化配置 `C=1.0`，提高 `max_iter` 只是给优化器足够的迭代预算；不做额外超参数搜索。先记录 validation accuracy，最终再和选中的小网络一起做测试。

baseline 如果已经不错，神经网络不一定会超过它。我们的目标是掌握可验证的训练过程，不预设网络必须获胜。


In [ ]:
baseline = LogisticRegression(C=1.0, max_iter=2000, random_state=SEED)
baseline.fit(X_train, y_train)
baseline_val_accuracy = accuracy_score(y_val, baseline.predict(X_val))
print(f"Logistic regression validation accuracy: {baseline_val_accuracy:.4f}")
# 保存线性分类器的数值参数；后面可直接由分数 argmax 复现其类别预测。
np.savez(OUT / "baseline_parameters.npz",
         coef=baseline.coef_, intercept=baseline.intercept_, classes=baseline.classes_)


### 7｜先固定比较协议，再动手训练

一个 **batch** 是一次更新使用的一小组样本；一个 **epoch** 是训练集完整走一遍。最后一个 batch 可能不足 64，交叉熵仍取这个 batch 的平均。

本次固定网络 `64→32→10`、CPU / float64、SGD（无 momentum、无 weight decay）、batch size 64、50 epochs，只比较学习率 **0.01 和 0.1**。同一个初始 `state_dict`，同一份每轮打乱的样本顺序，减少初始化和顺序对比较的干扰。

每轮末尾都重新评价完整 train / validation。只有 validation loss 严格变小才复制 checkpoint；相等保留较早轮次。两配置再按最佳 validation loss 选择，完全相等时保留列表中较早的学习率。不会根据测试结果修改这条规则。


In [ ]:
LEARNING_RATES = [0.01, 0.1]
EPOCHS, BATCH_SIZE = 50, 64

initial_model = make_model()
copy_numpy_params(initial_model, init_params(64, 32, 10, seed=SEED))
# state_dict 的 Tensor 可能仍引用模型数据；逐项 clone 得到真正的快照。
initial_state = {key: value.detach().clone() for key, value in initial_model.state_dict().items()}
order_rng = np.random.default_rng(SEED)
epoch_orders = [order_rng.permutation(len(train_idx)) for _ in range(EPOCHS)]
np.save(OUT / "epoch_orders.npy", np.stack(epoch_orders))
torch.save(initial_state, OUT / "initial_state.pt")
print("参数量:", sum(p.numel() for p in initial_model.parameters()))
assert sum(p.numel() for p in initial_model.parameters()) == 2410


### 8｜先学评价函数，再写一个 epoch

`model.train()` / `model.eval()` 设置模块运行模式；它们**不负责开关自动求导**。本书只有 Linear 与 ReLU，切换模式不会改变数值结果。以后遇到 Dropout / BatchNorm 时，模式才会影响这些层的行为。评价时仍明确调用 `eval()`，并用 `no_grad()` 关闭计算图，避免不必要的内存开销。

下面的 `evaluate` 不更新参数。函数返回普通 Python 数值，避免把计算图留在训练日志里。


In [ ]:
def evaluate(model, features, targets):
    model.eval()
    with torch.no_grad():
        logits = model(features)
        loss_value = criterion(logits, targets).item()
        accuracy = (logits.argmax(dim=1) == targets).double().mean().item()
    return loss_value, accuracy


# 验证“模式”和“是否记录梯度”是两件事。
initial_model.train()
train_mode_logits = initial_model(X_train_t[:4])
initial_model.eval()
with torch.no_grad():
    eval_mode_logits = initial_model(X_train_t[:4])
assert train_mode_logits.requires_grad and not eval_mode_logits.requires_grad
assert torch.equal(train_mode_logits.detach(), eval_mode_logits)
print("本结构在 train/eval 模式下数值一致，no_grad 关闭梯度追踪。")


In [ ]:
def train_one_epoch(model, optimizer, sample_order):
    model.train()  # evaluate 会切到 eval，因此每轮训练重新明确设为 train。
    for start in range(0, len(sample_order), BATCH_SIZE):
        indices = torch.tensor(sample_order[start:start + BATCH_SIZE], dtype=torch.long)
        xb, yb = X_train_t[indices], y_train_t[indices]
        optimizer.zero_grad(set_to_none=True)  # 清除上一 batch 的梯度。
        logits = model(xb)                     # 前向：得到原始类别分数。
        loss = criterion(logits, yb)           # 平均交叉熵。
        if not torch.isfinite(loss):
            raise FloatingPointError("loss 非有限：检查数据、梯度、学习率，并保留失败记录。")
        loss.backward()                       # 反向：计算参数梯度。
        optimizer.step()                      # 更新：θ ← θ - lr × grad。


### 9｜把 50 轮串起来，复制最佳状态

只写 `best_state = model.state_dict()` 不足以保留过去的最佳权重，因为其中的 Tensor 仍可能随之后的训练改变。下面明确 `.detach().clone()`。每轮完整训练集 loss 是“这一轮结束时同一个模型”的评价，而不是把不断变化的 batch loss 随手平均，因此可以和同一时刻的 validation loss 比较。

本例按预先固定的 50 轮运行完，再恢复其中最好的 checkpoint，**没有提前停止**。学习率过大或其他改动导致失败时，本例会抛错并停止，未完成轮次的日志不会自动保存。先把配置、已打印结果和完整 traceback 复制到自己的实验记录，再修复；不要偷偷换设置后继续使用同一个试验名称。


In [ ]:
def fit_one_learning_rate(lr):
    model = make_model()
    model.load_state_dict(initial_state)  # 两次都从相同数值出发。
    for key, value in model.state_dict().items():
        assert torch.equal(value, initial_state[key])
    optimizer = torch.optim.SGD(model.parameters(), lr=lr)
    history, best_loss, best_epoch, best_state = [], float("inf"), None, None
    for epoch, sample_order in enumerate(epoch_orders, start=1):
        train_one_epoch(model, optimizer, sample_order)
        train_loss, train_acc = evaluate(model, X_train_t, y_train_t)
        val_loss, val_acc = evaluate(model, X_val_t, y_val_t)
        history.append({"epoch": epoch, "lr": lr, "train_loss": train_loss,
                        "val_loss": val_loss, "train_accuracy": train_acc,
                        "val_accuracy": val_acc})
        if val_loss < best_loss:  # 严格小于：平局保留先出现的 epoch。
            best_loss, best_epoch = val_loss, epoch
            best_state = {key: value.detach().clone()
                          for key, value in model.state_dict().items()}
    assert best_state is not None
    for key, value in model.state_dict().items():
        assert value.data_ptr() != best_state[key].data_ptr()  # 不是别名。
    return {"lr": lr, "best_val_loss": best_loss, "best_epoch": best_epoch,
            "state": best_state, "history": pd.DataFrame(history)}


In [ ]:
runs = []
for lr in LEARNING_RATES:
    run = fit_one_learning_rate(lr)
    runs.append(run)
    run["history"].to_csv(OUT / f"history_lr_{lr:g}.csv", index=False)
    torch.save(run["state"], OUT / f"checkpoint_lr_{lr:g}.pt")
    print(f"lr={lr:g}: best epoch={run['best_epoch']}, val loss={run['best_val_loss']:.4f}")

# Python 的 min 遇到完全相同的 key 会保留列表中先出现的项。
chosen = min(runs, key=lambda run: run["best_val_loss"])
selection_table = pd.DataFrame([
    {"lr": r["lr"], "best_epoch": r["best_epoch"], "best_val_loss": r["best_val_loss"],
     "best_val_accuracy": r["history"].loc[r["history"]["epoch"] == r["best_epoch"], "val_accuracy"].iloc[0],
     "selected": r is chosen} for r in runs])
selection_table.to_csv(OUT / "validation_selection.csv", index=False)
display(selection_table)
print("选择规则是 val loss，不是表中的 val accuracy。")


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
for run in runs:
    h = run["history"]
    axes[0].plot(h["epoch"], h["train_loss"], label=f"lr={run['lr']} train")
    axes[0].plot(h["epoch"], h["val_loss"], linestyle="--", label=f"lr={run['lr']} validation")
    axes[1].plot(h["epoch"], h["train_accuracy"], label=f"lr={run['lr']} train")
    axes[1].plot(h["epoch"], h["val_accuracy"], linestyle="--", label=f"lr={run['lr']} validation")
axes[0].set(ylabel="Mean cross entropy", xlabel="Epoch", title="Loss")
axes[1].set(ylabel="Accuracy", xlabel="Epoch", title="Accuracy", ylim=(0, 1.02))
for ax in axes:
    ax.legend(fontsize=8)
    ax.grid(alpha=0.2)
fig.tight_layout()
fig.savefig(OUT / "training_curves.png", dpi=150)
plt.show()


**怎样读结果：** 学习率较小时下降可能慢；大一点通常学得快，但这不是普遍保证。train loss 下降而 validation loss 回升提示泛化差距变大，应按既定规则取早先的 checkpoint。loss 与 accuracy 不完全同步：分类是否正确没变，自信程度仍会改变交叉熵。只做了一个种子，不能据此宣称某学习率普遍更好。

### 10｜测试前写下冻结协议

到这里已经选择了预处理、网络结构、学习率和 checkpoint。保存这些决定后才进入测试。**冻结不是停止梯度计算的技术开关，而是实验决策：看到测试结果后不再用它改配置。** 若后续继续研究，要另外设计验证流程，不能把这个测试集变成新的调参集。


In [ ]:
frozen_config = {
    "dataset": "sklearn.load_digits", "split_seed": SEED,
    "split": "60/20/20 stratified indexed split; 40% heldout split equally",
    "train_size": len(train_idx), "val_size": len(val_idx), "test_size": len(test_idx),
    "preprocessor": "StandardScaler fit on train only; stored mean and scale",
    "architecture": [64, 32, 10], "activation": "ReLU", "dtype": "float64", "device": "cpu",
    "optimizer": "SGD", "momentum": 0.0, "weight_decay": 0.0,
    "batch_size": BATCH_SIZE, "epochs_budget": EPOCHS, "initialization_seed": SEED,
    "learning_rates_compared": LEARNING_RATES, "selected_lr": chosen["lr"],
    "selected_epoch": chosen["best_epoch"], "selected_val_loss": chosen["best_val_loss"],
    "selection_rule": "lowest validation loss; ties keep earlier epoch then earlier lr",
    "baseline": {"model": "LogisticRegression", "C": 1.0, "max_iter": 2000,
                 "validation_accuracy": float(baseline_val_accuracy)},
    "versions": {"python": platform.python_version(), "numpy": np.__version__,
                 "sklearn": sklearn.__version__, "torch": str(torch.__version__)},
    "comparison_tolerances": {"atol": ATOL, "rtol": RTOL},
    "test_not_used_for_selection": True,
}
(OUT / "frozen_config.json").write_text(json.dumps(frozen_config, indent=2, ensure_ascii=False), encoding="utf-8")
torch.save(chosen["state"], OUT / "selected_state.pt")
print("配置与 checkpoint 已冻结，尚未评价测试集。")


### 停下来独立练习 B（先不看测试）

在自己的结果说明里回答：

1. 两个学习率各在哪一轮 validation loss 最低？为什么不直接用第 50 轮？
2. 用自己的曲线解释“训练还没学好”和“训练好但泛化较差”的差别；曲线没有出现某种现象也要如实说。
3. 如果只写 `best_state=model.state_dict()`，为什么“最佳 checkpoint”可能偷偷变成最后一轮？
4. baseline 的 validation accuracy 是多少？选网络时为什么仍坚持提前约定的 validation loss？

<details><summary>提示</summary>

从 `validation_selection.csv` 与曲线取证。两边 loss 都高通常先检查优化和表达能力；train 低但 validation 高是另一类问题。Tensor 的引用与数值副本不同。baseline 是固定参照，网络选择规则则是先定好的另一件事，不能看哪个指标有利就改用哪个。

</details>

**本次完成标准：** 能独立写出一个 batch 的训练步骤、解释公平比较的控制变量、留下曲线和冻结配置。完成后再进入最终测试。


In [ ]:
# 独立练习区：把自己的解释写进个人 Notebook 的 Markdown 单元。
# TODO：至少引用一项数值和一段曲线作为解释的证据。


<a id="w3-10"></a>
## W3-10｜重载模型，再做一次最终评价

### 11｜保存的不只是权重

`state_dict` 保存网络参数，但不会自动保存本课的网络结构、数据划分和 StandardScaler。本书分别保存 JSON 配置、`.pt` 权重、`.npz` 划分及预处理。下面从文件重建新的模型，并用验证集的同一小批输入核对保存前后 logits。

`.pt` 只放参数 Tensor，使用 `weights_only=True` 读取；加载自己刚保存的文件即可。`eval()` 与 `no_grad()` 仍各司其职。[参考：PyTorch 保存与加载](https://docs.pytorch.org/tutorials/beginner/saving_loading_models.html)


In [ ]:
# 保存前的参考：恢复选中的最佳状态，不用训练最后一轮的模型代替它。
selected_model = make_model()
selected_model.load_state_dict(chosen["state"])
selected_model.eval()
with torch.no_grad():
    reference_logits = as_numpy(selected_model(X_val_t[:16]))
np.savez(OUT / "reload_reference.npz", raw_features=X_raw[val_idx[:16]], logits=reference_logits)

# 新对象 + 从磁盘读配置 / 预处理 / 权重，检查完整推理路径。
loaded_config = json.loads((OUT / "frozen_config.json").read_text(encoding="utf-8"))
with np.load(OUT / "digits_split_and_scaler.npz", allow_pickle=False) as saved:
    loaded_mean, loaded_scale = saved["mean"].copy(), saved["scale"].copy()
reloaded_model = make_model(*loaded_config["architecture"])
reloaded_model.load_state_dict(torch.load(OUT / "selected_state.pt", map_location="cpu", weights_only=True))
reloaded_model.eval()
reloaded_input = (X_raw[val_idx[:16]] - loaded_mean) / loaded_scale
with torch.no_grad():
    reloaded_logits = as_numpy(reloaded_model(torch.tensor(reloaded_input, dtype=DTYPE)))
np.testing.assert_allclose(reloaded_logits, reference_logits, atol=ATOL, rtol=RTOL)
print("保存前后 logits 最大绝对差:", np.max(np.abs(reloaded_logits - reference_logits)))


### 12｜可复制到新内核的最小重载单元

下面单元显式导入自己需要的包、重新查找根目录、读取所有文件、重新定义结构，**不依赖上面的模型或变量**。完整跑完保存步骤后，可把这一格单独复制进空白 Notebook 运行。这里用之前保留的 validation 样本做重载一致性核验，不触碰测试集。


In [ ]:
# 独立重载演示：先前必须已经生成 outputs/pytorch_tutorial/ 的文件。
from pathlib import Path
import json
import numpy as np
import torch

reload_root = next((p for p in [Path.cwd(), *Path.cwd().parents]
                    if (p / "course/bootcamp").is_dir()), None)
if reload_root is None:
    raise RuntimeError("请从本项目目录内运行这个单元。")
reload_dir = reload_root / "course/bootcamp/week3/outputs/pytorch_tutorial"
reload_cfg = json.loads((reload_dir / "frozen_config.json").read_text(encoding="utf-8"))
d_in, d_hidden, d_out = reload_cfg["architecture"]
fresh_model = torch.nn.Sequential(torch.nn.Linear(d_in, d_hidden), torch.nn.ReLU(),
                                  torch.nn.Linear(d_hidden, d_out)).double().cpu()
fresh_model.load_state_dict(torch.load(reload_dir / "selected_state.pt", map_location="cpu", weights_only=True))
with np.load(reload_dir / "digits_split_and_scaler.npz", allow_pickle=False) as pre:
    fresh_mean, fresh_scale = pre["mean"].copy(), pre["scale"].copy()
with np.load(reload_dir / "reload_reference.npz", allow_pickle=False) as ref:
    fresh_x = (ref["raw_features"] - fresh_mean) / fresh_scale
    expected_logits = ref["logits"].copy()
fresh_model.eval()
with torch.no_grad():
    fresh_logits = fresh_model(torch.tensor(fresh_x, dtype=torch.float64)).numpy()
np.testing.assert_allclose(fresh_logits, expected_logits, **reload_cfg["comparison_tolerances"])
print("独立重载成功；前 5 个预测：", fresh_logits.argmax(axis=1)[:5])


### 13｜现在才打开测试集

测试集只用于报告冻结系统的最终表现。网络使用磁盘重载后的权重和预处理，baseline 也使用保存的训练参数；两者评价同一组测试索引。报告 accuracy、每个数字的 precision / recall / F1 / support 和混淆矩阵。混淆矩阵的行是真实类别，列是预测类别。


In [ ]:
# 测试流程的入口：前面的训练与模型选择没有调用这些数据做评价。
assert loaded_config["test_not_used_for_selection"] is True
X_test = (X_raw[test_idx] - loaded_mean) / loaded_scale
y_test = y[test_idx]
with torch.no_grad():
    test_logits = as_numpy(reloaded_model(torch.tensor(X_test, dtype=DTYPE)))
net_test_pred = test_logits.argmax(axis=1)

with np.load(OUT / "baseline_parameters.npz", allow_pickle=False) as saved_baseline:
    baseline_scores = X_test @ saved_baseline["coef"].T + saved_baseline["intercept"]
    baseline_test_pred = saved_baseline["classes"][baseline_scores.argmax(axis=1)]
np.testing.assert_array_equal(baseline_test_pred, baseline.predict(X_test))

test_predictions = {"logistic_regression": baseline_test_pred, "selected_network": net_test_pred}
final_results = pd.DataFrame([
    {"model": name, "test_accuracy": accuracy_score(y_test, pred), "test_size": len(y_test)}
    for name, pred in test_predictions.items()])
final_results.to_csv(OUT / "final_test_results.csv", index=False)
pd.DataFrame({"original_index": test_idx, "true_label": y_test,
              "baseline_prediction": baseline_test_pred, "network_prediction": net_test_pred
             }).to_csv(OUT / "test_predictions.csv", index=False)
display(final_results)


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 4.5))
for ax, (name, predictions) in zip(axes, test_predictions.items()):
    report = pd.DataFrame(classification_report(
        y_test, predictions, labels=np.arange(10), output_dict=True, zero_division=0)).T
    report.to_csv(OUT / f"{name}_classification_report.csv", index_label="class_or_average")
    print(name, "— 每类 precision / recall / F1 / support")
    display(report)
    matrix = confusion_matrix(y_test, predictions, labels=np.arange(10))
    pd.DataFrame(matrix, index=np.arange(10), columns=np.arange(10)).to_csv(
        OUT / f"{name}_confusion_matrix.csv", index_label="true_label")
    ConfusionMatrixDisplay(matrix, display_labels=np.arange(10)).plot(ax=ax, colorbar=False)
    ax.set_title(name)
fig.tight_layout()
fig.savefig(OUT / "test_confusion_matrices.png", dpi=150)
plt.show()


### 14｜从错例里提出解释，而不是反过来调参

展示最多 10 个**所选网络**的错误样本。使用 `test_idx` 回到原始 8×8 图像，而不是把标准化后的特征误当原始像素。图上标真实类别、预测类别和原始索引；没有错误则如实显示提示。

先区分“看起来像另一个数字”和“仅凭当前证据还不清楚”。错例可以提出后续研究问题，但不能在本次实验里看着测试图改网络再宣称同一个测试结果仍独立。


In [ ]:
wrong_positions = np.flatnonzero(net_test_pred != y_test)
shown = wrong_positions[:10]
pd.DataFrame({"original_index": test_idx[wrong_positions], "true_label": y_test[wrong_positions],
              "prediction": net_test_pred[wrong_positions]}).to_csv(OUT / "network_errors.csv", index=False)
print(f"错误共 {len(wrong_positions)} 个，本次展示 {len(shown)} 个。")
if len(shown) > 0:
    fig, axes = plt.subplots(1, len(shown), figsize=(max(3, 1.8 * len(shown)), 2.5), squeeze=False)
    for ax, pos in zip(axes.ravel(), shown):
        ax.imshow(digits.images[test_idx[pos]], cmap="gray_r", vmin=0, vmax=16)
        ax.set_title(f"id={test_idx[pos]}\ntrue={y_test[pos]}, pred={net_test_pred[pos]}", fontsize=9)
        ax.axis("off")
    fig.tight_layout()
    fig.savefig(OUT / "network_error_examples.png", dpi=150)
    plt.show()
else:
    print("这次固定划分没有错误图；不能据此推断所有新数据都会正确。")


### 15｜把结果变成以后能理解的项目记录

下面生成演示结果说明。它能帮你找文件，但你的结业说明还需要写出自己的曲线解读、错例观察与局限。单次划分、单个随机种子、小型教学数据都限制了结论；digits 上的表现不能直接外推到手机照片或其他手写数字来源。


In [ ]:
readme_text = f"""# PyTorch 入门演示结果

来源：course/bootcamp/week3/02_pytorch_training_walkthrough.ipynb。
运行：按 docs/setup/DEEP_LEARNING.md 选择 ML · PyTorch (.venv-dl) 内核，重启并 Run All。
环境版本见 frozen_config.json；数据是 sklearn.load_digits，无需下载。

固定 seed={SEED}；训练/验证/测试样本数={len(train_idx)}/{len(val_idx)}/{len(test_idx)}。
StandardScaler 只拟合训练集。网络为 64→32→10，ReLU，CPU float64，SGD，batch=64，50 epochs。
只比较 lr=0.01 与 0.1，共用初始权重和每轮样本顺序。
按最低 validation loss 选 checkpoint，轮次平局保留较早轮；配置平局保留较早 lr。
选中 lr={chosen['lr']}，epoch={chosen['best_epoch']}，validation loss={chosen['best_val_loss']:.6f}。
测试仅在配置冻结后运行；测试结果不参与任何上述选择。

## 文件入口

- numpy_pytorch_comparison.csv：前向、loss、梯度与一步 SGD 更新的数值核对。
- training_curves.png、history_lr_*.csv：每轮结束时完整 train / validation 指标。
- validation_selection.csv、frozen_config.json：选择证据与冻结规则。
- initial_state.pt、checkpoint_lr_*.pt、selected_state.pt：初始、各配置最佳、最终权重。
- epoch_orders.npy：两次训练共用的每轮样本顺序。
- digits_split_and_scaler.npz：索引、训练均值和 scale；baseline_parameters.npz：基线数值参数。
- reload_reference.npz：独立重载核验所需输入与期望 logits。
- final_test_results.csv、test_predictions.csv：冻结模型的最终评价。
- *_classification_report.csv、*_confusion_matrix.csv：逐类指标与混淆矩阵。
- test_confusion_matrices.png、network_errors.csv：混淆图与全部网络错误索引。
- network_error_examples.png：最多 10 张错图，有错误时生成。

## 结果

{final_results.to_string(index=False)}

## 自己补写的复盘

1. 选定 checkpoint 附近，训练与验证曲线说明什么？引用实际数值。
2. 至少观察两个错误案例；没有错误时说明这一划分的局限。
3. 哪些现象证据充分，哪些解释只是猜测？
4. 局限：单个 seed、小型数据、固定一次划分；不能推出某学习率或模型普遍更好。

新内核重载：复制原 Notebook 的“可复制到新内核的最小重载单元”并单独运行。
"""
(OUT / "README.md").write_text(readme_text, encoding="utf-8")
print("结果目录：", OUT)
print("请打开结果 README，再完成自己的复盘。")


### 独立练习 C｜不看演示写一次“交接给未来的自己”

在 `work/intro-network/README.md` 写清：运行环境、数据划分、预处理规则、模型和参数量、选择标准、最终测试结果、至少两项局限。把本周练习、自写实现和证据放在同一项目内，引用演示时明确标出哪些是参考、哪些是你独立完成的。

然后新建一个空白 Notebook，仅凭保存的配置、权重、预处理和参考输入重载模型，断言 logits 一致。你应能解释为什么只保存 `selected_state.pt` 还不足以还原整个推理过程。

**W3-10 完成标准：** 能画出输入到损失的 shape，手写一个 batch 的训练步骤；说明梯度检查和框架对照验证了什么；给出测试评价与错例；从新内核复现推理。按[通用项目评分](../../ASSESSMENT.md#project-rubric)自评后再登记进度。

| 现象 | 先检查 | 原因 |
|---|---|---|
| `No module named torch` | [深度学习环境](../../../docs/setup/DEEP_LEARNING.md)、打印的 `sys.executable` | 装包的环境与当前内核可能不同 |
| `mat1 and mat2 ...` | 输入是否 64 维、Linear 权重 shape | 矩阵乘法维数不匹配 |
| `Double` / `Float` 不匹配 | 数据与模型是否都 float64 | 默认 Linear 常为 float32 |
| 标签类型报错 | `y.dtype == torch.long`，取值 0～9 | 本例交叉熵使用类别下标 |
| 框架对照失败 | 转置、mean、softmax、清零顺序 | 先定位差异再训练真实数据 |
| loss 不降 | 先检查单 batch 的步骤与输入、再看学习率 | 先排实现错误，不急着换复杂网络 |
| “最佳”权重等于最后一轮且异常 | checkpoint 是否 clone | 保存引用不是保存过去的数值 |
| 重载预测变了 | 同一结构、dtype、预处理、最佳 checkpoint | 权重只是完整系统的一部分 |

**后续材料按需读：** [本地 CS231n PyTorch Notebook](../../../library/stanford-cs231n-2025/unpacked/47b3a1b8-assignment2/PyTorch.ipynb)可对照 Tensor 与训练循环；其中其他结构不是本次追加作业。此处不增加 Adam、Dropout、BatchNorm、多 seed 搜索，之后按学习指南进入数学基础与系统训练阶段。
